In [ ]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))

env_file = ROOT_DIR / ".env"
if env_file.is_file():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

from eoa import (
    ExplanationLayer,
    SpecializedAgent,
    SQLiteSessionStore,
)

In [ ]:
incident_402 = {
    "prediction": {
        "model_name": "XGBoost_CoolerCondition_v2",
        "target_variable": "Condição do Resfriador Hidráulico",
        "predicted_value": "Eficiência Reduzida (Nível Crítico)",
        "confidence_or_probability": 0.86,
        "status_label": "Operação Degradada / Alerta Crítico",
    },
    "xai": {
        "method": "SHAP",
        "feature_attributions": {
            "PS1_Pressao_Entrada": 0.54,
            "EPS1_Potencia_Motor": 0.32,
            "TS2_Temp_Trocador": 0.14,
        },
    },
    "context": {
        "variables": {
            "PS1_Pressao_Entrada": "Bar, faixa normal de operação [2.0, 4.0] bar",
            "EPS1_Potencia_Motor": "kW, faixa normal de operação [1.5, 2.8] kW",
        },
        "thresholds": {"PS1_max": "4.0 bar", "EPS1_max": "3.0 kW"},
        "documents": {
            "POP_Resfriamento_04": "Seção 4: Se PS1 exceder 4.0 bar acompanhado por EPS1 > 3.0 kW, há provável cavitação no filtro de sucção. Reduzir carga hidráulica imediatamente."
        },
    },
}

In [ ]:
store = SQLiteSessionStore(db_path=ROOT_DIR / "examples" / "live_gemini_sessions.db")
chat_layer = ExplanationLayer.default_layer(session_store=store)
session_id = "sessao_real_turbina_01"
chat_layer.clear_session(session_id)

In [ ]:
res1 = chat_layer.explain(
    prediction=incident_402["prediction"],
    xai=incident_402["xai"],
    context=incident_402["context"],
    user_query="Resuma o problema detectado no resfriador.",
    session_id=session_id,
)
print("[TURNO 1]:\n", res1.narrative)

In [ ]:
res2 = chat_layer.explain(
    user_query="Se não pudermos parar a bomba agora, qual o risco para o motor elétrico (EPS1)?",
    session_id=session_id,
)
print("\n[TURNO 2 - FOLLOW-UP]:\n", res2.narrative)
print(f"\nScore Turno 2: {res2.validation_score}/10.0 | Mensagens no banco: {len(chat_layer.get_history(session_id))}")

In [ ]:
res3 = await chat_layer.aexplain(
    user_query="Me explica as variáveis que estamos tratando e o que elas representam, você tem alguma tool associada?",
    session_id=session_id,
)

print(f"\n[TURNO 3 - FOLLOW-UP (Async)] (Score: {res3.validation_score}/10.0):\n", res3.narrative)

In [ ]:
chat_layer.override_agent_prompt(
    name="explainer",
    prompt="""
Responda de forma extremamente objetiva e técnica à dúvida do operador, sem rodeios e sem encher linguiça.

Dúvida do operador:
{user_query}
""",
)

chat_layer.inspect()

In [ ]:
res4 = chat_layer.explain(
    user_query="O que é um motor de pressão?",
    session_id=session_id,
)

print(f"\n[TURNO 4 - Pós reescrita no prompt] (Score: {res4.validation_score}/10.0):\n", res4.narrative)

In [ ]:
chat_layer.reset_agent_prompt("explainer")


def soma_fake(a: int, b: int) -> int:
    """Multiplica os fatores a e b simulando uma soma fake customizada."""
    return a * b


matematico = SpecializedAgent(
    name="math",
    role="Matemática Fake",
    prompt="""
Você é um especialista em matemática fake.
A ferramenta soma_fake executou e calculou o seguinte resultado numérico: {tool_outputs}
Explique o resultado obtido e o conceito de cálculo fake para a dúvida: {user_query}
""",
)

matematico.register_tool(soma_fake)
chat_layer.register_agent(matematico)

chat_layer.inspect()

In [ ]:
res_com_tool = chat_layer.explain(
    user_query="Faça uma soma fake de 10 e 20 e apresente o resultado.",
    session_id=session_id,
)
math_step_a = next(s for s in res_com_tool.execution_trace if s.agent_name == "math")
print("=== CASO A: Pergunta Matemática ('soma fake de 10 e 20') ===")
print("Trace do Agente Math:", math_step_a.summary)
print("Resposta:\n", res_com_tool.narrative)

print("\n" + "=" * 75 + "\n")

res_sem_tool = chat_layer.explain(
    user_query="Explique brevemente o que é cavitação no filtro de sucção segundo o procedimento operacional.",
    session_id=session_id,
)
math_step_b = next(s for s in res_sem_tool.execution_trace if s.agent_name == "math")
print("=== CASO B: Pergunta Conceitual ('o que é cavitação') ===")
print("Trace do Agente Math:", math_step_b.summary)
print("Resposta:\n", res_sem_tool.narrative)

In [ ]:
history = chat_layer.get_history(session_id=session_id)

for msg in history:
    print(f"[{msg.role.upper()}]: {msg.content}\n" + "-" * 60)

In [ ]:
# Modo Zero-ML + Hidratação Progressiva de Contexto via Tools no CMA
# (Sem modelo de ML nem XAI: o ArtifactsAgent economiza 100% dos tokens pulando o LLM,
#  e o CMA usa tools sob demanda acumulando documentos na sessão!)

zero_ml_layer = ExplanationLayer.default_layer(session_store=store)
zero_ml_session = "sessao_zero_ml_progressiva"
zero_ml_layer.clear_session(zero_ml_session)

def buscar_manual_valvula(valvula_id: str) -> str:
    """Busca o procedimento operacional padrão de uma válvula hidráulica específica."""
    return f"Manual {valvula_id}: Pressão máxima de alívio é 4.0 bar. Em caso de bloqueio, abrir bypass manual."

cma_agent = zero_ml_layer.get_agent("context_manager")
cma_agent.register_tool(buscar_manual_valvula)

res_zero = zero_ml_layer.explain(
    user_query="Qual o procedimento operacional no manual da válvula V-102 em caso de bloqueio?",
    session_id=zero_ml_session,
)

aa_step = next(s for s in res_zero.execution_trace if s.agent_name == "artifacts")
print("Trace do ArtifactsAgent (Zero-ML):", aa_step.summary)
print("Documentos hidratados no AgentState.context:", zero_ml_layer.last_state.context.documents)
print("\nResposta Zero-ML Validada:\n", res_zero.narrative)

In [ ]:
state = chat_layer.last_state

print("=== 1. ARTEFATOS EM MEMÓRIA NO AGENTSTATE ===")
print("Prediction:", state.prediction.model_dump() if state.prediction else None)
print("Top Features XAI:", state.xai.top_features() if state.xai else None)
print("Context Variables:", list(state.context.variables.keys()) if state.context else None)
print("Context Documents:", list(state.context.documents.keys()) if state.context else None)

print("\n=== 2. BLACKBOARD INTERMEDIÁRIO DOS AGENTES ===")
print("Resumo CMA (context_summary):", (state.context_summary or "")[:160], "...")
print("Resumo AA (artifacts_summary):", (state.artifacts_summary or "")[:160], "...")
print("Especialistas (specialist_insights):", list(state.specialist_insights.keys()))

print("\n=== 3. RASTRO DE EXECUÇÃO E LATÊNCIA (EXECUTION TRACE) ===")
for step in res_sem_tool.execution_trace:
    print(f"  • {step.agent_name:<16} | {step.execution_time_ms:>8.1f} ms | {step.status:<10} | {step.summary}")

print("\n=== 4. AUDITORIAS GRAVADAS NO SQLITE (VALIDATION_LOGS) ===")
for log in store.get_validation_logs(session_id):
    print(f"  • [{log['timestamp']}] Score: {log['score']}/10.0 | Passed: {bool(log['passed'])}")

In [ ]:
cma = chat_layer.get_agent("context_manager")
print(cma.temperature)